# Detect objective-value outliers

Set `IDEAL_RESULT_SUBFOLDER` and `REAL_RESULT_SUBFOLDER` below. They may name different result folders. The notebook reads `treadmill_ideal_*`/`treadmill_fixed_*` from the ideal folder and `treadmill_realistic_*`/`treadmill_real_*` from the real folder, then detects outliers across the combined simulations. Speed and treadmill type are retained as descriptive labels.

Outliers are detected using the standard boxplot rule: values below Q1 − 1.5×IQR or above Q3 + 1.5×IQR. A complete participant–speed pair is excluded when either result is an objective outlier or has a solver status other than 0.

In [15]:
# Configuration
IDEAL_RESULT_SUBFOLDER = "0812_e300_sw"
REAL_RESULT_SUBFOLDER = "0812_e300_sw"
IQR_MULTIPLIER = 1.5
EXPORT_CSV = True
EXPORT_TRAJECTORY_CSV = True

In [16]:
import re
from pathlib import Path

import cloudpickle
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

# Work whether Jupyter starts in the repository root or in scripts/.
cwd = Path.cwd().resolve()
ROOT = cwd if (cwd / "result_HC").is_dir() else cwd.parent
if not (ROOT / "result_HC").is_dir():
    raise FileNotFoundError("Could not find the repository's result_HC directory.")

RESULT_ROOT = ROOT / "result_HC"
RESULT_RE = re.compile(r"treadmill_(ideal|realistic|fixed|real)_(\d+)_(\d+)\.pkl$")
MODEL_LABEL = {"ideal": "ideal", "fixed": "ideal", "realistic": "real", "real": "real"}

In [17]:
def decode_message(value):
    return value.decode(errors="replace") if isinstance(value, bytes) else str(value)


def load_result_summary(path, source_result_subfolder):
    participant_folder = path.relative_to(RESULT_ROOT).parts[0]
    match = RESULT_RE.fullmatch(path.name)
    if match is None:
        return None

    with path.open("rb") as handle:
        payload = cloudpickle.load(handle)

    # Biosym saves: ((states, globals), solver_info, settings).
    solver_info = payload[1]
    speed = float(f"{match.group(2)}.{match.group(3)}")
    objective = float(solver_info["obj_val"])
    status = int(solver_info.get("status", -999))
    return {
        "participant": participant_folder.upper(),
        "participant_number": int(participant_folder[1:]),
        "speed": speed,
        "model": MODEL_LABEL[match.group(1)],
        "result_stage": match.group(1),
        "source_result_subfolder": source_result_subfolder,
        "objective_value": objective,
        "finite_objective": np.isfinite(objective),
        "solver_status": status,
        "solver_success": status == 0,
        "solver_message": decode_message(solver_info.get("status_msg", "")),
        "path": str(path.relative_to(ROOT)),
    }


folder_by_model = {
    "ideal": IDEAL_RESULT_SUBFOLDER,
    "real": REAL_RESULT_SUBFOLDER,
}
rows = []
for wanted_model, result_subfolder in folder_by_model.items():
    paths = sorted(
        path for path in RESULT_ROOT.glob("p*/**/treadmill_*.pkl")
        if path.parent.name == result_subfolder
    )
    for path in paths:
        row = load_result_summary(path, result_subfolder)
        if row is not None and row["model"] == wanted_model:
            rows.append(row)
results = pd.DataFrame(rows)
if results.empty:
    raise RuntimeError(
        "No matching treadmill results found for ideal folder "
        f"{IDEAL_RESULT_SUBFOLDER!r} and real folder {REAL_RESULT_SUBFOLDER!r}."
    )

results = results.sort_values(["speed", "model", "participant_number"]).reset_index(drop=True)
print(
    f"Loaded {len(results)} results: ideal from {IDEAL_RESULT_SUBFOLDER!r}, "
    f"real from {REAL_RESULT_SUBFOLDER!r}."
)
display(results.groupby(["speed", "model"]).size().rename("n_results").unstack(fill_value=0))

RuntimeError: No ideal or realistic results found for '0812_e300_sw'.

In [14]:
def add_global_iqr_flags(data):
    data = data.copy()
    valid = data.loc[
        data["finite_objective"] & data["solver_status"].eq(0),
        "objective_value",
    ]
    q1 = valid.quantile(0.25)
    q3 = valid.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - IQR_MULTIPLIER * iqr
    upper = q3 + IQR_MULTIPLIER * iqr
    data["q1"] = q1
    data["q3"] = q3
    data["iqr"] = iqr
    data["lower_bound"] = lower
    data["upper_bound"] = upper
    data["objective_outlier"] = (~data["finite_objective"]) | (
        (data["objective_value"] < lower) | (data["objective_value"] > upper)
    )
    data["solver_failure"] = data["solver_status"].ne(0)
    row_excluded = data["objective_outlier"] | data["solver_failure"]
    data["exclude_pair"] = row_excluded.groupby(
        [data["participant"], data["speed"]]
    ).transform("max")
    data["needs_review"] = data["exclude_pair"]
    return data


evaluated = add_global_iqr_flags(results).reset_index(drop=True)
print(
    f"Global objective bounds: {evaluated['lower_bound'].iloc[0]:.6g} to "
    f"{evaluated['upper_bound'].iloc[0]:.6g} "
    f"(Q1={evaluated['q1'].iloc[0]:.6g}, Q3={evaluated['q3'].iloc[0]:.6g})"
)

review_columns = [
    "participant", "speed", "model", "objective_value",
    "lower_bound", "upper_bound", "objective_outlier",
    "solver_status", "solver_success", "solver_failure",
    "exclude_pair", "solver_message", "path",
]
outliers = evaluated.loc[evaluated["needs_review"], review_columns].sort_values(
    ["speed", "model", "objective_value"]
)
print(f"Flagged {len(outliers)} of {len(evaluated)} results for review.")
display(outliers)

KeyError: 'finite_objective'

In [5]:
# Objective distributions. Red diamonds are objective-value outliers.
sns.set_theme(style="whitegrid")
g = sns.catplot(
    data=evaluated, x="speed", y="objective_value", col="model",
    kind="box", sharey=False, showfliers=False, height=4, aspect=1.25,
)
for ax, model in zip(g.axes.flat, g.col_names):
    subset = evaluated[evaluated["model"] == model]
    order = sorted(subset["speed"].unique())
    x_positions = {speed: i for i, speed in enumerate(order)}
    ax.scatter(
        subset["speed"].map(x_positions), subset["objective_value"],
        color="0.25", alpha=0.65, s=25, zorder=3,
    )
    flagged = subset[subset["objective_outlier"]]
    ax.scatter(
        flagged["speed"].map(x_positions), flagged["objective_value"],
        color="crimson", marker="D", s=55, label="Outlier", zorder=4,
    )
    for _, row in flagged.iterrows():
        ax.annotate(row["participant"], (x_positions[row["speed"]], row["objective_value"]), xytext=(4, 4), textcoords="offset points")
    if len(flagged):
        ax.legend(frameon=False)
g.set_axis_labels("Speed (m/s)", "Final objective value")
g.set_titles("{col_name} treadmill")
plt.show()

NameError: name 'evaluated' is not defined

In [6]:
if EXPORT_CSV:
    folder_pair = f"ideal_{IDEAL_RESULT_SUBFOLDER}__real_{REAL_RESULT_SUBFOLDER}"
    safe_name = re.sub(r"[^A-Za-z0-9]+", "_", folder_pair).strip("_")
    output_path = RESULT_ROOT / f"objective_outliers_{safe_name}.csv"
    objective_outliers = evaluated.loc[evaluated["exclude_pair"]]
    objective_outliers.sort_values(["speed", "model", "participant_number"]).to_csv(output_path, index=False)
    print(f"Saved {output_path.relative_to(ROOT)}")

NameError: name 'evaluated' is not defined

In [16]:
# Export the trajectories used by downstream analysis notebooks.
# This creates the CSV loaded by 2_a_lower_limb_kinematics_kinetics.ipynb.
if EXPORT_TRAJECTORY_CSV:
    import subprocess
    import sys
    import tempfile

    folder_pair = f"ideal_{IDEAL_RESULT_SUBFOLDER}__real_{REAL_RESULT_SUBFOLDER}"
    safe_name = re.sub(r"[^A-Za-z0-9]+", "_", folder_pair).strip("_")
    trajectory_output_path = (
        RESULT_ROOT / f"simulation_results_101_points_{safe_name}.csv"
    )
    with tempfile.TemporaryDirectory() as temporary_directory:
        temporary_directory = Path(temporary_directory)
        exports = []
        for model, result_subfolder, model_names in [
            ("ideal", IDEAL_RESULT_SUBFOLDER, ["fixed_stage2", "fixed", "ideal"]),
            ("real", REAL_RESULT_SUBFOLDER, ["real", "realistic"]),
        ]:
            partial_output = temporary_directory / f"{model}.csv"
            command = [
                sys.executable,
                str(ROOT / "scripts" / "export_simulation_results_csv.py"),
                "--result-root", str(RESULT_ROOT),
                "--result-subdir", result_subfolder,
                "--n-points", "101",
                "--treadmill-models", *model_names,
                "--output", str(partial_output),
            ]
            subprocess.run(command, cwd=ROOT, check=True)
            exports.append(pd.read_csv(partial_output))
        pd.concat(exports, ignore_index=True).to_csv(trajectory_output_path, index=False)
    print(f"Saved {trajectory_output_path.relative_to(ROOT)}")

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


Loading model from cache: e2a4301806e799bbeb4d32058793715bf1d8e8c885caa37d6068023051f6509b_x32_eom_nomusclefit.cpkl
Loading model from cache: 827daca4aaa21e0234f11c64411793e67257b12d44f6297f908101e4bb56ee6d_x32_eom_nomusclefit.cpkl
Loading model from cache: 9dc5d7c2c5c2af5e4467f275fde91692488a61a9604903863db768cb5876e186_x32_eom_nomusclefit.cpkl
Replacing dynamic symbols in the EOM with the v_ states, this might take a while...
Replacing dynamic symbols in the EOM with the v_ states, this might take a while...
Loading model from cache: 24a7fd5a660a6f3d9819f448e5e38f013ea2ea14f3f796ac477986231c808343_x32_eom_nomusclefit.cpkl
Loading model from cache: 06b3223b80decceca6eed6b4585bbc7d787ef0b55231cd2455953f33e6ecfa98_x32_eom_nomusclefit.cpkl
Loading model from cache: 50916b866d22db257aabbd68e9544a19027f77811c154d13213f08dbcff987f1_x32_eom_nomusclefit.cpkl
Loading model from cache: 5157c89818d624dd20d1b2e87f4c88cf0f6879ed57358e040994edd68683b84d_x32_eom_nomusclefit.cpkl
Loading model from c